In [1]:
import os
import sys
import time
from ftplib import FTP
import subprocess
from datetime import datetime, timedelta
from concurrent.futures import ThreadPoolExecutor

# from config.config import aod_config

In [2]:


# --- Configuration ---
FTP_HOST = "ftp.ptree.jaxa.jp"
FTP_USER = "tr.hoanganh1124work_gmail.com" 
FTP_PASS = "SP+wari8"
BASE_DIR = "/pub/himawari/L2/ARP/031"

LOCAL_BASE = "/home/slow_data/Air_Quality/AOD"
PROCESS_SCRIPT = "/home/work1/projects/Air_Quality/AOD data/process_aod_data.py"


def get_local_files(local_path):
    """Get set of all processed files in local directory"""
    if not os.path.exists(local_path):
        return set()
    try:
        return set([f.removeprefix("aod_vietnam_").removesuffix(".tif") 
                   for f in os.listdir(local_path)])
    except Exception as e:
        print(f"⚠️ Error reading local directory {local_path}: {e}")
        return set()


def download_aod_for_hour(target_datetime, process_files=True):
    """
    Download AOD data for a specific hour
    
    Args:
        target_datetime (datetime): The specific hour to download data for
        process_files (bool): Whether to process .nc files after downloading (default: True)
    
    Returns:
        dict: Summary of the operation with keys:
            - success (bool): Whether the operation was successful
            - files_downloaded (int): Number of files downloaded
            - files_processed (int): Number of files processed
            - message (str): Status message
    
    Example:
        # Download data for October 11, 2025 at 15:00
        from datetime import datetime
        result = download_aod_for_hour(datetime(2025, 10, 11, 15, 0))
        print(result)
    """
    
    result = {
        'success': False,
        'files_downloaded': 0,
        'files_processed': 0,
        'message': ''
    }
    
    try:
        # Build paths
        ymd = target_datetime.strftime("%Y%m")
        dd = target_datetime.strftime("%d")
        hh = target_datetime.strftime("%H")
        remote_path = f"{BASE_DIR}/{ymd}/{dd}/{hh}/"
        local_path = os.path.join(LOCAL_BASE, ymd, dd, hh)
        
        print(f"🎯 Target: {target_datetime.strftime('%Y-%m-%d %H:%M:%S')}")
        print(f"📂 Remote path: {remote_path}")
        print(f"💾 Local path: {local_path}")
        
        # Create local directory
        os.makedirs(local_path, exist_ok=True)
        
        # Connect to FTP
        print(f"\n🔌 Connecting to FTP server...")
        with FTP(FTP_HOST, timeout=30) as ftp:
            ftp.login(FTP_USER, FTP_PASS)
            print(f"✅ Connected successfully")
            
            # Change to remote directory
            try:
                ftp.cwd(remote_path)
            except Exception as e:
                result['message'] = f"Remote directory not found: {remote_path}"
                print(f"❌ {result['message']}")
                return result
            
            # Get list of .nc files
            remote_files = ftp.nlst()
            remote_nc_files = [f for f in remote_files if f.endswith('.nc')]
            
            if not remote_nc_files:
                result['success'] = True
                result['message'] = "No .nc files found in remote directory"
                print(f"ℹ️ {result['message']}")
                return result
            
            print(f"📋 Found {len(remote_nc_files)} .nc files in remote directory")
            
            # Get already downloaded files
            local_files = get_local_files(local_path)
            
            # Filter files to download
            files_to_download = [f for f in remote_nc_files 
                               if f.removesuffix(".nc") not in local_files]
            
            if not files_to_download:
                result['success'] = True
                result['message'] = f"All {len(remote_nc_files)} files already exist locally"
                print(f"✔️ {result['message']}")
                return result
            
            print(f"\n📥 Downloading {len(files_to_download)} new files...")
            
            # Download files
            for i, file in enumerate(files_to_download, 1):
                local_file = os.path.join(local_path, file)
                
                try:
                    print(f"  [{i}/{len(files_to_download)}] Downloading {file}...", end=" ")
                    
                    with open(local_file, "wb") as f:
                        ftp.retrbinary(f"RETR {file}", f.write)
                    
                    print("✅")
                    result['files_downloaded'] += 1
                    
                    # Process the file if requested
                    if process_files:
                        try:
                            print(f"      Processing {file}...", end=" ")
                            subprocess.run(
                                ["python", PROCESS_SCRIPT, local_file],
                                check=True,
                                timeout=300,
                                capture_output=True
                            )
                            print("✅")
                            result['files_processed'] += 1
                        except subprocess.TimeoutExpired:
                            print("❌ Timeout (5 min)")
                        except Exception as proc_error:
                            print(f"❌ Error: {proc_error}")
                    
                except Exception as download_error:
                    print(f"❌ Error: {download_error}")
                    # Clean up partially downloaded file
                    if os.path.exists(local_file):
                        os.remove(local_file)
            
            result['success'] = True
            result['message'] = f"Downloaded {result['files_downloaded']} files, processed {result['files_processed']}"
            print(f"\n🎉 {result['message']}")
            
    except Exception as e:
        result['message'] = f"Error: {str(e)}"
        print(f"❌ {result['message']}")
    
    return result

In [3]:
target = datetime(2025, 12, 3, 4, 0)
result = download_aod_for_hour(target, process_files=False)
print(f"\nResult: {result}")

🎯 Target: 2025-12-03 04:00:00
📂 Remote path: /pub/himawari/L2/ARP/031/202512/03/04/
💾 Local path: /home/slow_data/Air_Quality/AOD/202512/03/04

🔌 Connecting to FTP server...
✅ Connected successfully
📋 Found 6 .nc files in remote directory

📥 Downloading 6 new files...
  [1/6] Downloading NC_H09_20251203_0410_L2ARP031_FLDK.02401_02401.nc... ✅
  [2/6] Downloading NC_H09_20251203_0430_L2ARP031_FLDK.02401_02401.nc... ✅
  [3/6] Downloading NC_H09_20251203_0420_L2ARP031_FLDK.02401_02401.nc... ✅
  [4/6] Downloading NC_H09_20251203_0440_L2ARP031_FLDK.02401_02401.nc... ✅
  [5/6] Downloading NC_H09_20251203_0400_L2ARP031_FLDK.02401_02401.nc... ✅
  [6/6] Downloading NC_H09_20251203_0450_L2ARP031_FLDK.02401_02401.nc... ✅

🎉 Downloaded 6 files, processed 0

Result: {'success': True, 'files_downloaded': 6, 'files_processed': 0, 'message': 'Downloaded 6 files, processed 0'}


In [4]:
import os
import sys
import rasterio
import numpy as np
import xarray as xr
import geopandas as gpd
import subprocess
from rasterio.mask import mask
from rasterio.transform import from_origin

# Define a standard "No Data" value (Transparent)
NODATA_VAL = -9999.0

def nc_to_geotiff(nc_file, output_path):
    print(f"Reading: {nc_file}")
    ds = xr.open_dataset(nc_file, decode_timedelta=True)
    
    # 1. Define variables to extract
    # Use a dictionary to map Variable Name -> Data Array
    vars_to_extract = ['AOT', 'AOT_uncertainty', 'AE', 'QA_flag', 'SSA', 'RF']
    data_map = {}

    lon = ds['longitude'].values
    lat = ds['latitude'].values
    
    # 2. CHECK FOR FLIP (The fix for the "White Shape" issue)
    # If latitude is increasing (e.g., -90 to 90), it is South-to-North.
    # GeoTIFF requires North-to-South (Top-Left origin).
    needs_flip = lat[0] < lat[-1]
    
    if needs_flip:
        print(" > Data is South-to-North. Flipping arrays...")

    # 3. Process each variable
    for v in vars_to_extract:
        if v in ds:
            val = ds[v].values
            
            # Remove time dimension if present (1, H, W) -> (H, W)
            if val.ndim == 3:
                val = val.squeeze()
            
            # FLIP DATA IF NEEDED
            if needs_flip:
                val = np.flipud(val)
                
            data_map[v] = val
        else:
            print(f"Warning: {v} not found in NetCDF. Filling with NoData.")
            # Create an empty array of the correct shape
            ref_shape = data_map.get('AOT', (len(lat), len(lon))).shape
            data_map[v] = np.full(ref_shape, np.nan)

    ds.close()

    # 4. Align Grid
    pixel_size = 0.05
    lon_start = np.floor(lon.min() / pixel_size) * pixel_size
    # GeoTIFF origin is always the Top-Left (North-West), so we use lat.max()
    lat_start = np.ceil(lat.max() / pixel_size) * pixel_size

    transform = from_origin(
        lon_start, lat_start, pixel_size, pixel_size
    )

    # 5. Create Profile with NODATA
    # We use the shape of the first variable ('AOT')
    height, width = data_map['AOT'].shape
    
    profile = {
        'driver': 'GTiff',
        'height': height,
        'width': width,
        'count': len(vars_to_extract),
        'dtype': 'float32',
        'crs': 'EPSG:4326',
        'transform': transform,
        'nodata': NODATA_VAL  # CRITICAL: Tells GIS this value is transparent
    }

    # 6. Write to File
    with rasterio.open(output_path, 'w', **profile) as dst:
        for i, name in enumerate(vars_to_extract):
            data = data_map[name]
            
            # Convert NaNs in the data to -9999
            data_filled = np.nan_to_num(data, nan=NODATA_VAL)
            
            dst.write(data_filled.astype('float32'), i + 1)
            dst.set_band_description(i + 1, name)

def crop_to_vietnam(input_tif, output_tif, vietnam_shapefile):
    print(f"Cropping: {input_tif}")
    with rasterio.open(input_tif) as src:
        shape = gpd.read_file(vietnam_shapefile)
        shape = shape.to_crs(src.crs)
        
        # MASK FUNCTION
        # nodata=NODATA_VAL: Ensures the outside is -9999 (not 0)
        # crop=True: Cuts the box to size
        try:
            out_image, out_transform = mask(src, shape.geometry, crop=True, nodata=NODATA_VAL)
        except ValueError:
            print("ERROR: Shape does not overlap raster. Check coordinates!")
            return

        out_meta = src.meta.copy()
        out_meta.update({
            "height": out_image.shape[1],
            "width": out_image.shape[2],
            "transform": out_transform,
            "nodata": NODATA_VAL # Update metadata to match
        })

    # Check if result is empty/all nodata
    valid_data = out_image[out_image != NODATA_VAL]
    if valid_data.size == 0:
        print("WARNING: Cropped image contains only NoData values.")

    with rasterio.open(output_tif, "w", **out_meta) as dest:
        dest.write(out_image)

In [7]:
nc_path = "/home/slow_data/Air_Quality/AOD/202512/03/04/NC_H09_20251203_0420_L2ARP031_FLDK.02401_02401.nc"

In [8]:
base_dir = os.path.dirname(nc_path)
filename = os.path.basename(nc_path).replace(".nc", "")
aod_full_path = os.path.join(base_dir, f"aod_full_{filename}.tif")
aod_vietnam_path = os.path.join(base_dir, f"aod_vietnam_{filename}.tif")

shapefile_path = "/home/work1/projects/Air_Quality/GADM_Vietnam/gadm41_VNM_0.shp"

# 1. Convert
nc_to_geotiff(nc_path, aod_full_path)

# 2. Crop
if os.path.exists(aod_full_path):
    crop_to_vietnam(aod_full_path, aod_vietnam_path, shapefile_path)

    # CLEANUP
    # IMPORTANT: I suggest commenting out the 'os.remove' lines 
    # until you verify the fix works, so you don't lose data.
    os.remove(nc_path) 
    os.remove(aod_full_path)

Reading: /home/slow_data/Air_Quality/AOD/202512/03/04/NC_H09_20251203_0420_L2ARP031_FLDK.02401_02401.nc
Cropping: /home/slow_data/Air_Quality/AOD/202512/03/04/aod_full_NC_H09_20251203_0420_L2ARP031_FLDK.02401_02401.tif


In [ ]:
tif = "/home/slow_data/Air_Quality/AOD/202509/11/15/aod_vietnam_NC_H09_20250911_1520_L2ARP031_FLDK.02401_02401.tif"